In [ ]:
import io, re, sys, subprocess
import numpy as np
import pandas as pd

try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx"])
    import networkx as nx

SEED = 42
N_PERM = 10_000

EDGES_PATH = None   # set to a path to skip the upload dialog
POSTS_PATH = None

if EDGES_PATH and POSTS_PATH:
    df_edges = pd.read_csv(EDGES_PATH, encoding="utf-8-sig")
    df_posts = pd.read_csv(POSTS_PATH, encoding="utf-8-sig")
else:
    from google.colab import files
    print(">>> Upload EDGES CSV (source,target)")
    up = files.upload()
    df_edges = pd.read_csv(io.BytesIO(up[list(up.keys())[0]]), encoding="utf-8-sig")
    print(">>> Upload POSTS CSV (User_SId, Text, classification, hate_class)")
    up = files.upload()
    df_posts = pd.read_csv(io.BytesIO(up[list(up.keys())[0]]), encoding="utf-8-sig")


def clean_cell(x):
    x = str(x).strip()
    if x.lower() in {"nan", "none", "null"}:
        return ""
    return re.sub(r"\s+", " ", x.replace("ي", "ی").replace("ك", "ک"))


for d in (df_edges, df_posts):
    for c in d.columns:
        if d[c].dtype == object:
            d[c] = d[c].apply(clean_cell)

df_posts = df_posts[df_posts["Text"].astype(str).str.strip() != ""].reset_index(drop=True)

# ---------------------------------------------------------------------
# normalize labels (same mapping as the training script)
# ---------------------------------------------------------------------
LABEL_MAP = {
    "عادی": "normal", "عادي": "normal", "normal": "normal",
    "توهین‌آمیز": "offensive", "توهین آمیز": "offensive", "توهین_آمیز": "offensive",
    "توهیننآمیز": "offensive", "توهینآمیز": "offensive", "توهين‌آميز": "offensive",
    "نفرت‌آمیز": "hateful", "نفرت آمیز": "hateful", "نفرت_آمیز": "hateful",
    "نفرتآمیز": "hateful", "تنفرآمیز": "hateful", "تنفر‌آمیز": "hateful",
    "تنفر آمیز": "hateful", "نفرت‌انگیز": "hateful",
}
unknown = set(df_posts["classification"].astype(str).str.strip()) - set(LABEL_MAP)
if unknown:
    print(f"WARNING: unmapped labels become 'normal': {sorted(unknown)[:10]}")
df_posts["lab"] = df_posts["classification"].astype(str).str.strip().map(
    lambda s: LABEL_MAP.get(s, "normal"))
df_posts["User_SId"] = df_posts["User_SId"].astype(str)

# ---------------------------------------------------------------------
# graph
# ---------------------------------------------------------------------
users = set(df_posts["User_SId"])
sc, dc = df_edges.columns[0], df_edges.columns[1]

G = nx.DiGraph()
G.add_nodes_from(users)
dup = 0
for _, r in df_edges.iterrows():
    s, t = str(r[sc]), str(r[dc])
    if s in users and t in users and s != t:
        if G.has_edge(s, t):
            dup += 1
        G.add_edge(s, t)

U = G.to_undirected()

N = G.number_of_nodes()
E_dir = G.number_of_edges()
E_und = U.number_of_edges()
degs = dict(U.degree())
avg_deg = 2 * E_und / N
max_deg = max(degs.values())
n_comp = nx.number_connected_components(U)
density = nx.density(U)

print("\n" + "=" * 70)
print("REPORT - copy everything below")
print("=" * 70)
print(f"  rows in edge file        : {len(df_edges)}")
print(f"  duplicate directed edges : {dup}")
print(f"  nodes                    : {N}")
print(f"  directed edges (in graph): {E_dir}")
print(f"  undirected edges         : {E_und}")
print(f"  average degree           : {avg_deg:.2f}")
print(f"  maximum degree           : {max_deg}")
print(f"  connected components     : {n_comp}")
print(f"  density                  : {density:.4f}")
print("        sure avg degree and density in the paper match the numbers")
print("        above (they are recomputed here from the corrected graph).")

# ---------------------------------------------------------------------
# per-user attributes
# ---------------------------------------------------------------------
per_user = (df_posts.groupby("User_SId")["lab"].value_counts()
            .unstack(fill_value=0)
            .reindex(columns=["normal", "offensive", "hateful"], fill_value=0))
per_user["total"] = per_user.sum(axis=1)
per_user = per_user.reindex(sorted(users), fill_value=0)

has_hate = (per_user["hateful"] > 0).astype(int).to_dict()
has_harm = ((per_user["hateful"] > 0) | (per_user["offensive"] > 0)).astype(int).to_dict()

# ---------------------------------------------------------------------
# communities
# ---------------------------------------------------------------------
comms = nx.community.louvain_communities(U, seed=SEED)
comms = sorted(comms, key=len, reverse=True)
node2c = {n: i for i, c in enumerate(comms) for n in c}
mod = nx.community.modularity(U, comms)

print(f"  communities : {len(comms)}")
print(f"  modularity  : {mod:.3f}")

rows = []
for i, c in enumerate(comms, start=1):
    sub = df_posts[df_posts["User_SId"].isin(c)]
    n_cont = len(sub)
    if n_cont == 0:
        rows.append({"community": f"C{i}", "users": len(c), "contents": 0,
                     "offensive_pct": 0.0, "hate_pct": 0.0, "users_with_hate_pct": 0.0})
        continue
    off = (sub["lab"] == "offensive").sum() / n_cont * 100
    hat = (sub["lab"] == "hateful").sum() / n_cont * 100
    uwh = np.mean([has_hate.get(u, 0) for u in c]) * 100
    rows.append({"community": f"C{i}", "users": len(c), "contents": n_cont,
                 "offensive_pct": round(off, 1), "hate_pct": round(hat, 1),
                 "users_with_hate_pct": round(uwh, 1)})
tbl = pd.DataFrame(rows)
print()
print(tbl.to_string(index=False))
print(f"\n  offensive rate range : {tbl['offensive_pct'].min():.1f}% .. {tbl['offensive_pct'].max():.1f}%")
print(f"  hate rate range      : {tbl['hate_pct'].min():.1f}% .. {tbl['hate_pct'].max():.1f}%")
print(f"  totals check         : users {tbl['users'].sum()}, contents {tbl['contents'].sum()}")

print("\n--- LaTeX for Table 2 (paste into the paper) ---")
for _, r in tbl.iterrows():
    print(f"{r['community']} & {r['users']} & {r['contents']:,} & "
          f"{r['offensive_pct']} & {r['hate_pct']} & {r['users_with_hate_pct']} \\\\")

# ---------------------------------------------------------------------
# assortativity + permutation test
# ---------------------------------------------------------------------
def attr_stats(attr, name):
    nx.set_node_attributes(U, attr, "a")
    r = nx.attribute_assortativity_coefficient(U, "a")
    same = np.mean([attr[u] == attr[v] for u, v in U.edges()]) * 100
    vals = np.array([attr[n] for n in U.nodes()])
    idx = {n: i for i, n in enumerate(U.nodes())}
    ee = np.array([[idx[u], idx[v]] for u, v in U.edges()])
    rng = np.random.RandomState(SEED)
    null = np.empty(N_PERM)
    for k in range(N_PERM):
        p = rng.permutation(vals)
        null[k] = np.mean(p[ee[:, 0]] == p[ee[:, 1]]) * 100
    pval = (np.sum(null >= same) + 1) / (N_PERM + 1)
    print(f"\n  [{name}]")
    print(f"    assortativity coefficient : {r:.3f}")
    print(f"    same-attribute edges      : {same:.1f}%")
    print(f"    null expectation (mean)   : {null.mean():.1f}%")
    print(f"    permutation p-value       : {'< 0.001' if pval < 0.001 else f'{pval:.4f}'}"
          f"  ({N_PERM} shuffles)")
    return r, same, null.mean(), pval


attr_stats(has_hate, "produced at least one hateful content")
attr_stats(has_harm, "produced at least one offensive or hateful content")



In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd

try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx"])
    import networkx as nx

EDGES_PATH = ""
POSTS_PATH = ""
SEED = 42
RESOLUTIONS = [0.6, 0.8, 1.0, 1.2, 1.5, 2.0]

for p in (EDGES_PATH, POSTS_PATH):
    if not os.path.exists(p):
        d = os.path.dirname(p) or "."
        raise FileNotFoundError(f"not found: {p}\nin {d}: {sorted(os.listdir(d))[:40]}")

df_edges = pd.read_csv(EDGES_PATH, encoding="utf-8-sig")
df_posts = pd.read_csv(POSTS_PATH, encoding="utf-8-sig")


def nid(x):
    s = str(x).strip()
    return s[:-2] if s.endswith(".0") and s[:-2].isdigit() else s


users = set(df_posts["User_SId"].map(nid))
sc, dc = df_edges.columns[0], df_edges.columns[1]

G = nx.DiGraph()
G.add_nodes_from(users)
for s, t in zip(df_edges[sc].map(nid), df_edges[dc].map(nid)):
    if s in users and t in users and s != t:
        G.add_edge(s, t)
U = G.to_undirected()

print("=" * 72)
print("COMMUNITY CONNECTIVITY CHECK")
print("=" * 72)
print(f"graph: {U.number_of_nodes()} nodes, {U.number_of_edges()} undirected edges")
print(f"graph connected components: {nx.number_connected_components(U)}")


def audit(comms, label):
    """Report how many communities are internally disconnected."""
    n_bad = 0
    detail = []
    for i, c in enumerate(sorted(comms, key=len, reverse=True), 1):
        sub = U.subgraph(c)
        k = nx.number_connected_components(sub)
        if k > 1:
            n_bad += 1
            sizes = sorted((len(x) for x in nx.connected_components(sub)), reverse=True)
            detail.append((i, len(c), k, sizes))
    mod = nx.community.modularity(U, comms)
    print(f"\n  {label}")
    print(f"    communities        : {len(comms)}")
    print(f"    modularity         : {mod:.3f}")
    print(f"    internally split   : {n_bad} / {len(comms)}")
    for i, size, k, sizes in detail:
        print(f"       C{i}: {size} users -> {k} disconnected pieces {sizes[:6]}")
    return len(comms), mod, n_bad


print("\n" + "-" * 72)
print("LOUVAIN ACROSS RESOLUTIONS")
print("-" * 72)
rows = []
for r in RESOLUTIONS:
    comms = nx.community.louvain_communities(U, seed=SEED, resolution=r)
    n, mod, bad = audit(comms, f"resolution = {r}")
    rows.append({"resolution": r, "communities": n, "modularity": round(mod, 3),
                 "disconnected": bad})

print("\n" + "-" * 72)
print("SUMMARY")
print("-" * 72)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

clean = summary[summary["disconnected"] == 0]
if len(clean):
    best = clean.sort_values("modularity", ascending=False).iloc[0]
    print(f"\n  resolutions with ALL communities connected: "
          f"{sorted(clean['resolution'].tolist())}")
    print(f"  highest modularity among those: resolution={best['resolution']}, "
          f"{int(best['communities'])} communities, modularity={best['modularity']}")
else:
    print("\n  no tested resolution gives fully connected communities with Louvain.")

# ---------------------------------------------------------------------
# Leiden (guarantees connected communities) if available
# ---------------------------------------------------------------------
print("\n" + "-" * 72)
print("LEIDEN (guarantees internally connected communities)")
print("-" * 72)
try:
    import igraph as ig
    import leidenalg
    nodes = list(U.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    g = ig.Graph(n=len(nodes),
                 edges=[(idx[u], idx[v]) for u, v in U.edges()], directed=False)
    for r in RESOLUTIONS:
        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            resolution_parameter=r, seed=SEED)
        comms = [set(nodes[i] for i in grp) for grp in part]
        audit(comms, f"Leiden resolution = {r}")
except ImportError:
    print("  leidenalg not installed. To compare, run:")
    print("     !pip install -q leidenalg python-igraph")
    print("  and re-run this cell.")

print("\n" + "=" * 72)
print("WHAT TO DO WITH THIS")
print("=" * 72)
print("""  If 'internally split' is 0 at your chosen resolution, the paper's
  claim that a community forms a connected subgraph holds, and nothing
  needs to change.

  If it is not 0, two options:
    (a) switch to Leiden, which guarantees connected communities, and
        report the new numbers; or
    (b) keep Louvain and soften the wording so the paper does not claim
        internal connectivity.""")
